# Simplified OSAR Control Analysis - Dabest Comparison
Quick analysis of w1118 controls across light intensities using dabest

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import OSAR and dabest
import sys
sys.path.append(r'C:\Users\user\Documents\GitHub\OSAR')
import osar
import dabest

print("Libraries imported successfully!")
print(f"OSAR version: {osar.__version__}")

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 19.06it/s]

Numba compilation complete!
Libraries imported successfully!
OSAR version: 0.23.7


## 2. Find Cross Folders and Process Controls

In [2]:
# Set directory
osar_directory = Path(r"D:\ACC Lab Dropbox\ACC Lab\Nicole Lee\DATA\OSAR")
output_dir = Path("C:/Users/user/Documents/GitHub/OSAR-Nicole-version/analysis_output")
output_dir.mkdir(exist_ok=True)

# Find first 5 cross folders
cross_folders = []
for folder in osar_directory.iterdir():
    if folder.is_dir() and 'x' in folder.name:
        cross_folders.append(folder)
        if len(cross_folders) >= 5:  # Only take first 5
            break

print(f"Found {len(cross_folders)} cross folders to analyze:")
for folder in cross_folders:
    print(f"  - {folder.name}")

Found 5 cross folders to analyze:
  - MB018B x ACR
  - MB018B x Chrimson2
  - MB077B x ACR
  - MB077B x Chrimson2
  - MB082C x ACR


## 3. Process Controls with OSAR

In [3]:
all_control_data = []

for cross_folder in cross_folders:
    print(f"\nProcessing {cross_folder.name}...")
    
    # Extract driver and responder
    parts = cross_folder.name.split(' x ')
    if len(parts) >= 2:
        driver = parts[0].strip()
        responder_part = parts[1].strip()
        
        # Check for known responders
        responder = None
        for resp in ['ACR', 'Chrimson2']:
            if resp in responder_part:
                responder = resp
                break
        
        if responder is None:
            responder = responder_part.split()[0]
        
        print(f"  Driver: {driver}, Responder: {responder}")
        
        try:
            # Use OSAR to process the data
            osar_analysis = osar.osar(str(cross_folder), driver=driver)
            
            # Extract control data (w1118 flies)
            results_df = osar_analysis.results
            control_results = results_df[results_df['genotype'].str.contains('w1118', na=False)]
            
            if not control_results.empty:
                # Add cross information
                control_results = control_results.copy()
                control_results['cross_name'] = cross_folder.name
                control_results['driver'] = driver
                control_results['responder'] = responder
                
                all_control_data.append(control_results)
                print(f"  ✓ Extracted {len(control_results)} control flies")
            else:
                print(f"  ✗ No w1118 controls found")
                
        except Exception as e:
            print(f"  ✗ Error: {e}")

# Combine all data
if all_control_data:
    combined_controls = pd.concat(all_control_data, ignore_index=True)
    print(f"\n✓ Total control flies: {len(combined_controls)}")
    print(f"✓ Crosses: {combined_controls['cross_name'].nunique()}")
    print(f"✓ Light intensities: {combined_controls['light_intensity'].unique()}")
else:
    print("\n✗ No control data found")


Processing MB018B x ACR...
  Driver: MB018B, Responder: ACR
Creating borders for each fly...
Done.

Processing CSV 43 of 43
Summarising results for all flies...
All done.
  ✓ Extracted 393 control flies

Processing MB018B x Chrimson2...
  Driver: MB018B, Responder: Chrimson2
Creating borders for each fly...
Done.

Processing CSV 56 of 56
Summarising results for all flies...
All done.
  ✓ Extracted 624 control flies

Processing MB077B x ACR...
  Driver: MB077B, Responder: ACR
Creating borders for each fly...
Done.

Processing CSV 55 of 55
Summarising results for all flies...
All done.
  ✓ Extracted 404 control flies

Processing MB077B x Chrimson2...
  Driver: MB077B, Responder: Chrimson2
Creating borders for each fly...
Done.

Processing CSV 52 of 52
Summarising results for all flies...
All done.
  ✓ Extracted 456 control flies

Processing MB082C x ACR...
  Driver: MB082C, Responder: ACR
Creating borders for each fly...
Done.

Processing CSV 44 of 44
Summarising results for all flies..

## 4. Prepare Data for Dabest Analysis

In [ ]:
# Create a combined grouping variable
dabest_data['control'] = dabest_data['genotype'].str.split().str[-1]
dabest_data['control_intensity'] = dabest_data['control'].astype('str') + '_' + dabest_data['light_intensity'].astype('str')

dabest_data

,responder,light_intensity,genotype,pi_smoothed_baseline_corrected,cross_name,control
0,ACR,Eighth,w1118 x MB018B,-0.007146,MB018B x ACR,MB018B
1,ACR,Eighth,w1118 x MB018B,-0.057047,MB018B x ACR,MB018B
2,ACR,Eighth,w1118 x MB018B,0.111217,MB018B x ACR,MB018B
3,ACR,Eighth,w1118 x MB018B,-0.061613,MB018B x ACR,MB018B
4,ACR,Eighth,w1118 x MB018B,-0.067838,MB018B x ACR,MB018B
...,...,...,...,...,...,...
2016,ACR,Full,w1118 x ACR,-0.204918,MB082C x ACR,ACR
2017,ACR,Full,w1118 x ACR,-0.237334,MB082C x ACR,ACR
2018,ACR,Full,w1118 x ACR,-0.178718,MB082C x ACR,ACR
2019,ACR,Full,w1118 x ACR,-0.158830,MB082C x ACR,ACR


In [13]:
if 'combined_controls' in locals() and not combined_controls.empty:
    # Show data summary
    print("Control data summary:")
    summary = combined_controls.groupby(['responder', 'light_intensity']).agg({
        'pi_smoothed_baseline_corrected': ['count', 'mean', 'std']
    }).round(3)
    display(summary)
    
    # Create comparison groups for dabest
    # We'll compare PI values across light intensities for each responder
    dabest_data = combined_controls[[
        'responder', 'light_intensity', 'genotype', 
        'pi_smoothed_baseline_corrected', 'cross_name'
    ]].copy()
    
    # Create a combined grouping variable
    dabest_data['control'] = dabest_data['genotype'].str.split().str[-1]
    dabest_data['control_intensity'] = dabest_data['control'].astype('str') + '_' + dabest_data['light_intensity'].astype('str')
    
    print(f"\nData prepared for dabest analysis:")
    print(f"Groups: {dabest_data['control_intensity'].unique()}")
    
    display(dabest_data.head())
else:
    print("No data available for analysis")

Control data summary:


pi_smoothed_baseline_corrected              
                                                   count   mean    std
responder light_intensity                                             
ACR       Eighth                                     226 -0.002  0.191
          Quarter                                    239 -0.038  0.214
          Half                                       238 -0.079  0.186
          Full                                       238 -0.092  0.170
Chrimson2 Eighth                                     270 -0.012  0.180
          Quarter                                    270 -0.035  0.191
          Half                                       270 -0.028  0.168
          Full                                       270 -0.016  0.192


Data prepared for dabest analysis:
Groups: ['MB018B_Eighth' 'ACR_Eighth' 'ACR_Quarter' 'MB018B_Quarter' 'MB018B_Half'
 'ACR_Half' 'MB018B_Full' 'ACR_Full' 'Chrimson2_Eighth'
 'Chrimson2_Quarter' 'Chrimson2_Half' 'Chrimson2_Full' 'MB077B_Eighth'
 'MB077B_Quarter' 'MB077B_Half' 'MB077B_Full']


,responder,light_intensity,genotype,pi_smoothed_baseline_corrected,cross_name,control,control_intensity
0,ACR,Eighth,w1118 x MB018B,-0.007146,MB018B x ACR,MB018B,MB018B_Eighth
1,ACR,Eighth,w1118 x MB018B,-0.057047,MB018B x ACR,MB018B,MB018B_Eighth
2,ACR,Eighth,w1118 x MB018B,0.111217,MB018B x ACR,MB018B,MB018B_Eighth
3,ACR,Eighth,w1118 x MB018B,-0.061613,MB018B x ACR,MB018B,MB018B_Eighth
4,ACR,Eighth,w1118 x MB018B,-0.067838,MB018B x ACR,MB018B,MB018B_Eighth


In [14]:
dabest_data

,responder,light_intensity,genotype,pi_smoothed_baseline_corrected,cross_name,control,control_intensity
0,ACR,Eighth,w1118 x MB018B,-0.007146,MB018B x ACR,MB018B,MB018B_Eighth
1,ACR,Eighth,w1118 x MB018B,-0.057047,MB018B x ACR,MB018B,MB018B_Eighth
2,ACR,Eighth,w1118 x MB018B,0.111217,MB018B x ACR,MB018B,MB018B_Eighth
3,ACR,Eighth,w1118 x MB018B,-0.061613,MB018B x ACR,MB018B,MB018B_Eighth
4,ACR,Eighth,w1118 x MB018B,-0.067838,MB018B x ACR,MB018B,MB018B_Eighth
...,...,...,...,...,...,...,...
2016,ACR,Full,w1118 x ACR,-0.204918,MB082C x ACR,ACR,ACR_Full
2017,ACR,Full,w1118 x ACR,-0.237334,MB082C x ACR,ACR,ACR_Full
2018,ACR,Full,w1118 x ACR,-0.178718,MB082C x ACR,ACR,ACR_Full
2019,ACR,Full,w1118 x ACR,-0.158830,MB082C x ACR,ACR,ACR_Full


## 5. Dabest Analysis - Compare Light Intensities Within Each Responder

In [ ]:
if 'dabest_data' in locals():
    # Analysis for each responder separately
    responders = dabest_data['responder'].unique()
    
    for responder in responders:
        print(f"\n{'='*50}")
        print(f"DABEST Analysis for {responder} Controls")
        print(f"{'='*50}")
        
        # Filter data for this responder
        resp_data = dabest_data[dabest_data['responder'] == responder].copy()
        
        if len(resp_data) < 10:  # Need minimum data
            print(f"Not enough data for {responder} ({len(resp_data)} flies)")
            continue
            
        # Get available light intensities for this responder
        intensities = sorted(resp_data['light_intensity'].unique())
        print(f"Light intensities available: {intensities}")
        
        if len(intensities) < 2:
            print(f"Need at least 2 light intensities for comparison")
            continue
        
        try:
            # Create dabest object comparing all intensities
            # Use first intensity as control, compare others to it
            control_intensity = intensities[0]
            test_intensities = intensities[1:]
            
            # Create comparison groups
            comparison_groups = [control_intensity] + test_intensities
            
            print(f"Comparing {test_intensities} vs {control_intensity} (control)")
            
            # Create dabest object
            dabest_obj = dabest.load(data=resp_data, 
                                   x='light_intensity', 
                                   y='pi_smoothed_baseline_corrected',
                                   idx=comparison_groups)
            
            # Calculate effect sizes
            effect_sizes = dabest_obj.hedges_g
            
            # Create plot
            fig = effect_sizes.plot(fig_size=(12, 8),
                                  contrast_label=f'PI Difference\n({responder} Controls)',
                                  swarm_label='PI Value')
            
            plt.suptitle(f'{responder} Control PI Across Light Intensities', 
                        fontsize=14, fontweight='bold')
            plt.tight_layout()
            
            # Save plot
            plt.savefig(output_dir / f'{responder}_control_dabest.png', 
                       dpi=300, bbox_inches='tight')
            plt.show()
            
            # Print statistical results
            print(f"\nStatistical Results for {responder}:")
            results_df = effect_sizes.statistical_tests
            display(results_df)
            
        except Exception as e:
            print(f"Error in dabest analysis for {responder}: {e}")
            
            # Fallback: simple comparison plot
            plt.figure(figsize=(10, 6))
            sns.boxplot(data=resp_data, x='light_intensity', y='pi_smoothed_baseline_corrected')
            sns.swarmplot(data=resp_data, x='light_intensity', y='pi_smoothed_baseline_corrected', 
                         color='black', alpha=0.6, size=3)
            plt.title(f'{responder} Control PI Across Light Intensities')
            plt.ylabel('PI Value')
            plt.xticks(rotation=45)
            plt.axhline(y=0, color='red', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.savefig(output_dir / f'{responder}_control_simple.png', 
                       dpi=300, bbox_inches='tight')
            plt.show()
else:
    print("No data available for dabest analysis")

## 6. Cross-Responder Comparison

In [ ]:
if 'dabest_data' in locals() and len(responders) > 1:
    print(f"\n{'='*50}")
    print(f"Comparing Controls Between Responders")
    print(f"{'='*50}")
    
    # For each light intensity, compare between responders
    intensities = dabest_data['light_intensity'].unique()
    
    for intensity in intensities:
        intensity_data = dabest_data[dabest_data['light_intensity'] == intensity].copy()
        
        if len(intensity_data['responder'].unique()) < 2:
            print(f"Not enough responders for {intensity} comparison")
            continue
            
        print(f"\nAnalyzing {intensity} intensity:")
        
        try:
            # Create dabest comparison between responders
            responder_list = sorted(intensity_data['responder'].unique())
            
            dabest_resp = dabest.load(data=intensity_data,
                                    x='responder',
                                    y='pi_smoothed_baseline_corrected',
                                    idx=responder_list)
            
            effect_sizes_resp = dabest_resp.hedges_g
            
            # Plot
            fig = effect_sizes_resp.plot(fig_size=(10, 6),
                                        contrast_label=f'PI Difference\n({intensity} Light)',
                                        swarm_label='PI Value')
            
            plt.suptitle(f'Control Comparison: {intensity} Light Intensity', 
                        fontsize=14, fontweight='bold')
            plt.tight_layout()
            
            plt.savefig(output_dir / f'responder_comparison_{intensity}.png', 
                       dpi=300, bbox_inches='tight')
            plt.show()
            
            # Print results
            print(f"Statistical results for {intensity}:")
            results_df = effect_sizes_resp.statistical_tests
            display(results_df)
            
        except Exception as e:
            print(f"Error in responder comparison for {intensity}: {e}")
else:
    print("Not enough data for cross-responder comparison")

## 7. Summary of Control Variability

In [ ]:
if 'combined_controls' in locals():
    print(f"\n{'='*60}")
    print(f"SUMMARY: Control Group Variability Analysis")
    print(f"{'='*60}")
    
    # Calculate variability metrics
    variability_summary = combined_controls.groupby(['responder', 'light_intensity']).agg({
        'pi_smoothed_baseline_corrected': ['count', 'mean', 'std', 'min', 'max']
    }).round(4)
    
    print("\nControl group statistics by responder and light intensity:")
    display(variability_summary)
    
    # Check for U-shaped patterns in means
    print("\nMean PI values across light intensities:")
    pivot_means = combined_controls.pivot_table(
        values='pi_smoothed_baseline_corrected',
        index='responder',
        columns='light_intensity',
        aggfunc='mean'
    ).round(4)
    
    display(pivot_means)
    
    # Simple U-shape detection
    print("\nU-shape pattern check (visual inspection):")
    for responder in pivot_means.index:
        pi_values = pivot_means.loc[responder].dropna()
        if len(pi_values) >= 4:
            # Check if ends are higher than middle (simple U-check)
            intensities = pi_values.index.tolist()
            if 'Eighth' in intensities and 'Full' in intensities and 'Quarter' in intensities and 'Half' in intensities:
                eighth = pi_values['Eighth']
                quarter = pi_values['Quarter']
                half = pi_values['Half']
                full = pi_values['Full']
                
                u_score = (eighth + full) - (quarter + half)
                print(f"  {responder}: U-score = {u_score:.4f} {'(U-shaped)' if u_score > 0.05 else '(Not U-shaped)'}")
    
    print(f"\n📁 All plots saved to: {output_dir}")
    
    # Final recommendation
    print(f"\n💡 RECOMMENDATION:")
    print(f"   - If controls show large U-scores (>0.1), your experimental U-patterns may be artifacts")
    print(f"   - If controls are relatively flat (U-scores <0.05), your experimental patterns are likely real")
    print(f"   - Use these control baselines for your delta calculations")
else:
    print("No control data found for analysis")